# Standalone Eagle-I / Whisker spike MP4

Run all cells in order in **spwxr_network_new**. No variables or imports from another notebook are required. Inputs are local; nothing is downloaded or rebuilt. Output: a 288-frame H.264 MP4 (72 seconds at 4 fps), plus count/provenance sidecars.

This reproduces the supplied exploratory definitions: hourly Eagle-I maxima forward-filled to 15 minutes; positive changes >= county positive-change p90 and 50 customers; Whisker sensor-event sums in a trailing hour >= positive-activity p90 and above the full-window median. Threshold reference: **May 8–15, 2024**; movie: **May 10–12**. Whisker includes the same five frequency/sag/swell files, **not** PowerOutages or THD.

**Interpretation limits:** hourly Eagle-I data do not acquire 15-minute resolution by forward filling, increases are not necessarily new physical outages, and sensor-event sums are not unique sensor counts. To reproduce the original, missing slots are forward-filled/zero-filled; white means no flag under that processing, not verified no impact or reporting coverage. “Both” means coincident flags, not equal severity, causation, or demonstrated commensurability. Counts are restricted to mapped CONUS counties so they match the animation.

Paths, FFmpeg choice, preview, frame cap, and overwrite behavior are configurable below. A cached Parquet panel is preferred for bounded reads; set `EAGLE_INPUT_MODE='annual_csv'` to read the original annual source in chunks. Cached-panel preprocessing can differ from the earlier raw annual workflow. The preferred basemap is the original notebook's cached TIGER geometry, with a local generalized Census geometry fallback (reported explicitly).

In [ ]:
# 1. Define Runtime Configuration and File Paths
from pathlib import Path

PROJECT_ROOT = Path('/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience')
EAGLE_DIR = Path('/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/EAGLE-I')
WHISKER_DIR = Path('/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/Whisker_Labs_Data_March2026/2024-05-08 to 2024-05-15 Event')
EVENT_START_TEXT = '2024-05-08 00:00:00'
EVENT_END_TEXT = '2024-05-15 23:59:59'
SPIKE_START_TEXT = '2024-05-10 00:00:00'
SPIKE_END_TEXT = '2024-05-12 23:45:00'
SPIKE_FREQ = '15min'
SPIKE_Q = 0.90
EAGLE_MIN_DELTA = 50.0
WHISKER_WINDOW = '1h'
EAGLE_INPUT_MODE = 'parquet'  # 'annual_csv' reproduces original raw annual ingestion.
EAGLE_PARQUET = EAGLE_DIR / 'processed/eaglei_county_15min_panel_2024_2025.parquet'
EAGLE_CSV = EAGLE_DIR / 'eaglei_outages_2024.csv'
COUNTY_CANDIDATES = [
    PROJECT_ROOT / 'notebooks/_cache/tl_2024_us_county/tl_2024_us_county.shp',
    EAGLE_DIR / 'processed/county_geometry_cache/cb_2024_us_county_500k.shp',
]
EVENT_FILES = ['Swells_Outside_CBEMA.json', 'Swells_Inside_CBEMA.json',
               'Sags_Outside_CBEMA.json', 'Sags_Inside_CBEMA.json', 'Frequency.json']
OUTPUT_DIR = Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/output")
SPIKE_OUT = OUTPUT_DIR / 'may2024_eagle_whisker_SPIKES_may10_12.mp4'
FPS = 4
DPI = 120
MAX_FRAMES = None  # Set to 4 for a quick encoding smoke test; None exports all frames.
OVERWRITE = False
SHOW_PREVIEW = True
DISPLAY_VIDEO = True
FFMPEG_PATH = None  # Optional absolute executable; auto-detect system/conda/imageio otherwise.

In [ ]:
# 2. Import Dependencies and Validate FFmpeg Availability
import json
import os
import shutil
import subprocess
import sys
from datetime import timezone

import geopandas as gpd
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Video, FileLink, display
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Patch

EVENT_START = pd.Timestamp(EVENT_START_TEXT, tz='UTC')
EVENT_END = pd.Timestamp(EVENT_END_TEXT, tz='UTC')
SPIKE_START = pd.Timestamp(SPIKE_START_TEXT, tz='UTC')
SPIKE_END = pd.Timestamp(SPIKE_END_TEXT, tz='UTC')
assert EVENT_START <= SPIKE_START <= SPIKE_END <= EVENT_END
assert 0 < SPIKE_Q < 1 and FPS > 0 and DPI > 0
if MAX_FRAMES is not None and MAX_FRAMES < 1:
    raise ValueError('MAX_FRAMES must be None or a positive integer.')
ffmpeg_candidates = [FFMPEG_PATH, os.environ.get('IMAGEIO_FFMPEG_EXE'),
                     shutil.which('ffmpeg'), str(Path(sys.executable).parent / 'ffmpeg')]
FFMPEG = next((str(p) for p in ffmpeg_candidates if p and Path(p).is_file() and os.access(p, os.X_OK)), None)
if FFMPEG is None:
    try:
        import imageio_ffmpeg
        FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
    except (ImportError, RuntimeError) as exc:
        raise RuntimeError('FFmpeg required: install ffmpeg in this conda environment, or set FFMPEG_PATH; no automatic installation.') from exc
mpl.rcParams['animation.ffmpeg_path'] = FFMPEG
encoder_info = subprocess.run([FFMPEG, '-hide_banner', '-encoders'], capture_output=True, text=True, check=True)
if 'libx264' not in encoder_info.stdout:
    raise RuntimeError('Selected FFmpeg lacks libx264; select an H.264-capable build.')
print('Interpreter:', sys.executable)
print('FFmpeg:', FFMPEG)


def normalize_fips(series):
    out = series.astype('string').str.strip().str.replace(r'\.0$', '', regex=True).str.zfill(5)
    return out.where(out.str.fullmatch(r'\d{5}', na=False))

In [ ]:
# 3. Load County Geometry and Prepare CONUS Basemap
COUNTY_SHP = next((p for p in COUNTY_CANDIDATES if p.exists()), None)
if COUNTY_SHP is None:
    raise FileNotFoundError('No local county shapefile found. Update COUNTY_CANDIDATES; no download is attempted.')
counties_movie = gpd.read_file(COUNTY_SHP)
if counties_movie.crs is None:
    raise ValueError('County geometry has no declared CRS; specify the correct source CRS first.')
counties_movie = counties_movie.to_crs('EPSG:4326')
counties_movie['county_fips'] = normalize_fips(counties_movie['GEOID'])
counties_movie['STATEFP'] = counties_movie['STATEFP'].astype('string').str.zfill(2)
counties_movie = counties_movie.dropna(subset=['county_fips', 'geometry'])
counties_movie = counties_movie.loc[~counties_movie.geometry.is_empty].copy()
if counties_movie['county_fips'].duplicated().any():
    raise ValueError('Duplicate county FIPS in geometry; dissolve/reconcile before use.')
conus = counties_movie.loc[~counties_movie['STATEFP'].isin({'02','15','60','66','69','72','78'})].copy()
conus = conus.reset_index(drop=True)
print({'county_source': str(COUNTY_SHP), 'CONUS_counties': len(conus)})

In [ ]:
# 4. Load and Normalize Eagle-I Outage Data
# Select ONE source; never merge annual files with daily-granular holdings.
def normalize_eagle_chunk(chunk, time_col, fips_col, out_col):
    block = pd.DataFrame({
        'time': pd.to_datetime(chunk[time_col], errors='coerce', utc=True),
        'county_fips': normalize_fips(chunk[fips_col]),
        'eagle_customers_out': pd.to_numeric(chunk[out_col], errors='coerce'),
    })
    block = block.loc[block['time'].between(EVENT_START, EVENT_END)].dropna(subset=['county_fips'])
    block['slot'] = block['time'].dt.floor(SPIKE_FREQ)
    return block.groupby(['slot', 'county_fips'], as_index=False)['eagle_customers_out'].max()


if EAGLE_INPUT_MODE == 'parquet':
    if not EAGLE_PARQUET.exists():
        raise FileNotFoundError('Cached panel missing. Choose annual_csv explicitly; no panel rebuild is attempted.')
    eagle_source = EAGLE_PARQUET
    # Bounded read: only event dates and required columns, not the full 58M-row panel.
    block = pd.read_parquet(
        EAGLE_PARQUET, columns=['time', 'county_fips', 'customers_out'],
        filters=[('time', '>=', EVENT_START.tz_localize(None).to_pydatetime()),
                 ('time', '<=', EVENT_END.tz_localize(None).to_pydatetime())],
    )
    eagle_15min = normalize_eagle_chunk(block, 'time', 'county_fips', 'customers_out')
    del block
elif EAGLE_INPUT_MODE == 'annual_csv':
    if not EAGLE_CSV.exists():
        raise FileNotFoundError(EAGLE_CSV)
    eagle_source = EAGLE_CSV
    columns = pd.read_csv(EAGLE_CSV, nrows=0).columns
    aliases = {c.lower().replace(' ', '_'): c for c in columns}
    time_col = next((aliases[k] for k in ['run_start_time', 'time', 'timestamp'] if k in aliases), None)
    fips_col = next((aliases[k] for k in ['fips_code', 'county_fips', 'fips'] if k in aliases), None)
    out_col = next((aliases[k] for k in ['customers_out', 'customers_affected'] if k in aliases), None)
    if None in (time_col, fips_col, out_col):
        raise KeyError(f'Unrecognized annual Eagle-I schema: {list(columns)}')
    pieces = []
    for chunk in pd.read_csv(EAGLE_CSV, usecols=[time_col, fips_col, out_col], dtype={fips_col: 'string'}, chunksize=250_000):
        piece = normalize_eagle_chunk(chunk, time_col, fips_col, out_col)
        if not piece.empty:
            pieces.append(piece)
    if not pieces:
        raise ValueError('No Eagle-I observations in the event window.')
    eagle_15min = pd.concat(pieces).groupby(['slot', 'county_fips'], as_index=False)['eagle_customers_out'].max()
else:
    raise ValueError('EAGLE_INPUT_MODE must be parquet or annual_csv.')

if eagle_15min.empty:
    raise ValueError('No Eagle-I observations found.')
print({'eagle_source': str(eagle_source), '15min_county_rows': len(eagle_15min),
       'counties': eagle_15min['county_fips'].nunique(),
       'first_UTC': str(eagle_15min['slot'].min()), 'last_UTC': str(eagle_15min['slot'].max())})

In [ ]:
# 5. Load and Normalize Whisker Event Data
records = []
for filename in EVENT_FILES:
    path = WHISKER_DIR / filename
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open(encoding='utf-8') as handle:
        payload = json.load(handle)
    event_rows = payload.get('features', []) if isinstance(payload, dict) else payload
    if not isinstance(event_rows, list):
        raise ValueError(f'Unrecognized Whisker structure: {path}')
    for item in event_rows:
        row = item.get('properties', item)
        loc = row.get('Location') or {}
        records.append({
            'event_family': path.stem,
            'start_utc': pd.to_datetime(row.get('StartTime'), errors='coerce', utc=True),
            'sensors': pd.to_numeric(row.get('Sensors'), errors='coerce'),
            'lat': pd.to_numeric(loc.get('Latitude'), errors='coerce'),
            'lon': pd.to_numeric(loc.get('Longitude'), errors='coerce'),
            'county_fips': row.get('county_fips'),
        })
whisker_events_df = pd.DataFrame(records)
if whisker_events_df.empty:
    raise ValueError('No Whisker events loaded.')
whisker_events_df = whisker_events_df.dropna(subset=['start_utc', 'lat', 'lon'])
whisker_events_df = whisker_events_df.loc[whisker_events_df['start_utc'].between(EVENT_START, EVENT_END)].copy()
whisker_events_df['sensors'] = whisker_events_df['sensors'].replace([np.inf, -np.inf], np.nan).fillna(1.0)
if (whisker_events_df['sensors'] < 0).any():
    raise ValueError('Negative Whisker sensor counts found.')
whisker_events_df['county_fips'] = normalize_fips(whisker_events_df['county_fips'])
whisker_events_df = whisker_events_df.reset_index(drop=True)
print('Whisker events by family:', whisker_events_df.groupby('event_family').size().to_dict())

In [ ]:
# 6. Assign Whisker Events to Counties (Spatial Join Fallback)
missing = whisker_events_df['county_fips'].isna()
if missing.any():
    points = gpd.GeoDataFrame(
        whisker_events_df.loc[missing, ['lat', 'lon']],
        geometry=gpd.points_from_xy(whisker_events_df.loc[missing, 'lon'], whisker_events_df.loc[missing, 'lat']),
        crs='EPSG:4326',
    )
    joined = gpd.sjoin(points, counties_movie[['county_fips', 'geometry']], how='left', predicate='within')
    if joined.index.duplicated().any():
        raise ValueError('A Whisker point maps to multiple county polygons; reconcile geometry before summing sensors.')
    whisker_events_df.loc[missing, 'county_fips'] = joined['county_fips'].reindex(whisker_events_df.index[missing])
whisker_unmapped = int(whisker_events_df['county_fips'].isna().sum())
whisker_outside_conus = int((whisker_events_df['county_fips'].notna() & ~whisker_events_df['county_fips'].isin(conus['county_fips'])).sum())
whisker_events_df = whisker_events_df.loc[whisker_events_df['county_fips'].isin(conus['county_fips'])].copy()
eagle_outside_conus = int((~eagle_15min['county_fips'].isin(conus['county_fips'])).sum())
eagle_15min = eagle_15min.loc[eagle_15min['county_fips'].isin(conus['county_fips'])].copy()
if whisker_events_df.empty or eagle_15min.empty:
    raise ValueError('No mapped CONUS observations in one of the input sources.')
print({'Whisker_unmapped_events': whisker_unmapped, 'Whisker_outside_mapped_CONUS': whisker_outside_conus,
       'Eagle_hourly_rows_outside_mapped_CONUS': eagle_outside_conus,
       'Whisker_mapped_events': len(whisker_events_df)})

In [ ]:
# 7. Build 15-Minute Time Grids and Event-Referenced Threshold Window
frame_times = pd.date_range(SPIKE_START, SPIKE_END, freq=SPIKE_FREQ)
full_times = pd.date_range(EVENT_START.floor('h'), EVENT_END.floor(SPIKE_FREQ), freq=SPIKE_FREQ)
if not len(frame_times) or not frame_times.isin(full_times).all():
    raise ValueError('Movie frames must be contained in the threshold-reference time grid.')
print({'movie_frames': len(frame_times), 'threshold_reference_slots': len(full_times),
       'movie_seconds_at_configured_fps': len(frame_times) / FPS})

In [ ]:
# 8. Compute Eagle-I Change-Referenced Spike Flags
# Reproduction mode: same .ffill().fillna(0) behavior as the supplied movie.
# This is not a coverage-aware production outage detector.
eagle_observed = (
    eagle_15min.pivot_table(
        index='slot',
        columns='county_fips',
        values='eagle_customers_out',
        aggfunc='max',
    )
    .sort_index()
)
print(
    'Missing 15-minute county values before filling:',
    int(eagle_observed.reindex(full_times).isna().sum().sum()),
)
eagle_wide = eagle_observed.reindex(full_times).ffill().fillna(0.0)
eagle_delta = eagle_wide.diff().clip(lower=0.0).fillna(0.0)


def _county_q_threshold(df, q):
    return df.where(df > 0).quantile(q)


eagle_thr = _county_q_threshold(eagle_delta, SPIKE_Q).fillna(np.inf)
eagle_spike = (eagle_delta >= eagle_thr) & (eagle_delta >= EAGLE_MIN_DELTA)
print('Counties with any Eagle-I spike in full reference window:', int(eagle_spike.any().sum()))

In [ ]:
# 9. Compute Whisker Rolling-Activity Spike Flags
wev = whisker_events_df.copy()
wev['slot'] = wev['start_utc'].dt.floor(SPIKE_FREQ)
whisker_wide = (wev.groupby(['slot', 'county_fips'])['sensors'].sum()
                .unstack('county_fips').reindex(full_times).fillna(0.0))
whisker_roll = whisker_wide.rolling(WHISKER_WINDOW).sum()
whisker_thr = _county_q_threshold(whisker_roll, SPIKE_Q).fillna(np.inf)
whisker_base = whisker_roll.median()
whisker_spike = ((whisker_roll >= whisker_thr) & (whisker_roll > whisker_base)
                 & (whisker_roll > 0))
print('Counties with any Whisker spike in full reference window:', int(whisker_spike.any().sum()))

In [ ]:
# 10. Combine Spike Classes and Compute Agreement Diagnostics
all_counties = sorted(set(eagle_spike.columns) | set(whisker_spike.columns))
E = eagle_spike.reindex(columns=all_counties, fill_value=False).reindex(frame_times, fill_value=False)
W = whisker_spike.reindex(columns=all_counties, fill_value=False).reindex(frame_times, fill_value=False)
spike_class = E.astype(np.uint8) + 2 * W.astype(np.uint8)
counts = pd.DataFrame({
    'eagle_only': (spike_class == 1).sum(axis=1),
    'whisker_only': (spike_class == 2).sum(axis=1),
    'both': (spike_class == 3).sum(axis=1),
})
n_e, n_w, n_b = int(E.to_numpy().sum()), int(W.to_numpy().sum()), int((E & W).to_numpy().sum())
jaccard = n_b / max(n_e + n_w - n_b, 1)
eagle_counts = counts['eagle_only'] + counts['both']
whisker_counts = counts['whisker_only'] + counts['both']
ts_corr = eagle_counts.corr(whisker_counts) if eagle_counts.std() > 0 and whisker_counts.std() > 0 else np.nan
assert E.index.equals(W.index) and E.columns.equals(W.columns)
assert spike_class.isin([0, 1, 2, 3]).all().all()
assert int(counts['both'].sum()) == n_b
assert int((counts['eagle_only'] + counts['both']).sum()) == n_e
assert int((counts['whisker_only'] + counts['both']).sum()) == n_w
print({'frames': len(frame_times), 'eagle_spike_county_frames': n_e,
       'whisker_spike_county_frames': n_w, 'both_county_frames': n_b,
       'jaccard': jaccard, 'count_timeseries_correlation': ts_corr})
display(counts.head())

In [ ]:
# 11. Render Animated County Spike Map with Time-Series Panel
class_colors = {1: '#e31a1c', 2: '#1f78b4', 3: '#6a3d9a'}
class_labels = {1: 'Eagle-I spike only', 2: 'Whisker spike only', 3: 'Both (coincident flags)'}
cmap = mcolors.ListedColormap(['#f7f7f7', class_colors[1], class_colors[2], class_colors[3]])
norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)
legend_handles = [Patch(facecolor=class_colors[k], label=class_labels[k]) for k in (1, 2, 3)]


def create_movie_figure():
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_axes([0.05, 0.30, 0.90, 0.64])
    ax_ts = fig.add_axes([0.08, 0.06, 0.86, 0.18])
    for key, class_id in [('eagle_only', 1), ('whisker_only', 2), ('both', 3)]:
        ax_ts.plot(counts.index, counts[key], color=class_colors[class_id],
                   lw=2.0 if class_id == 3 else 1.3, label=class_labels[class_id])
    ax_ts.set_ylabel('# spike counties')
    ax_ts.set_xlabel('Time (UTC)')
    ax_ts.grid(alpha=0.2)
    ax_ts.legend(loc='upper left', fontsize=8, ncol=3)
    ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d\n%H:%M', tz=timezone.utc))
    cursor = ax_ts.axvline(frame_times[0], color='k', lw=1.0)
    # Plot polygons once, then update their scalar colors. Exploding first makes
    # the polygon-to-county order explicit and avoids costly per-frame spatial joins.
    polygons = conus[['county_fips', 'geometry']].explode(index_parts=False).reset_index(drop=True)
    polygons.plot(ax=ax, color='#f7f7f7', linewidth=0.1, edgecolor='#bdbdbd')
    collection = ax.collections[-1]
    if len(collection.get_paths()) != len(polygons):
        plt.close(fig)
        raise ValueError('Unexpected polygon rendering order/count; cannot safely animate county colors.')
    collection.set_cmap(cmap)
    collection.set_norm(norm)
    ax.set_xlim(-125, -66)
    ax.set_ylim(24, 50)
    ax.set_axis_off()
    ax.legend(handles=legend_handles, loc='lower left', title=f'County spikes (q{int(SPIKE_Q * 100)})')

    def draw(i):
        t = frame_times[i]
        values = spike_class.loc[t].reindex(polygons['county_fips']).fillna(0).to_numpy(dtype=int)
        collection.set_array(values)
        c = counts.loc[t]
        ax.set_title('May 2024: Eagle-I vs Whisker change-referenced spikes by county\n'
                     f'{t:%Y-%m-%d %H:%M UTC} | Eagle-only={c.eagle_only}  '
                     f'Whisker-only={c.whisker_only}  Both={c.both}')
        cursor.set_xdata([t, t])
        return collection, cursor

    draw(0)
    return fig, draw


if SHOW_PREVIEW:
    preview_fig, preview_draw = create_movie_figure()
    plt.show()
    plt.close(preview_fig)

In [ ]:
# 12. Export MP4 and Emit Run Summary
export_frame_count = len(frame_times) if MAX_FRAMES is None else min(int(MAX_FRAMES), len(frame_times))
output_path = SPIKE_OUT if MAX_FRAMES is None else SPIKE_OUT.with_name(SPIKE_OUT.stem + '_preview.mp4')
summary_path = output_path.with_suffix('.summary.json')
counts_path = output_path.with_suffix('.counts.csv')
partial_path = output_path.with_name(output_path.stem + '.partial.mp4')
if not OVERWRITE and any(p.exists() for p in [output_path, summary_path, counts_path, partial_path]):
    raise FileExistsError('Output already exists. Change SPIKE_OUT or explicitly set OVERWRITE=True.')
output_path.parent.mkdir(parents=True, exist_ok=True)
fig, draw = create_movie_figure()
anim = FuncAnimation(fig, draw, frames=export_frame_count, interval=1000 / FPS,
                     blit=False, repeat=False, cache_frame_data=False)
writer = FFMpegWriter(fps=FPS, codec='libx264', bitrate=2400,
                     metadata={'title': 'Eagle-I / Whisker county spikes, May 2024'},
                     extra_args=['-pix_fmt', 'yuv420p', '-movflags', '+faststart',
                                 '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2'])


def report_progress(current, total):
    if current % 48 == 0 or current + 1 == total:
        print(f'Encoding frame {current + 1}/{total}', flush=True)


try:
    anim.save(str(partial_path), writer=writer, dpi=DPI, progress_callback=report_progress)
    partial_path.replace(output_path)
finally:
    plt.close(fig)
export_counts = counts.iloc[:export_frame_count]
export_e = int((export_counts['eagle_only'] + export_counts['both']).sum())
export_w = int((export_counts['whisker_only'] + export_counts['both']).sum())
export_b = int(export_counts['both'].sum())
export_ec = export_counts['eagle_only'] + export_counts['both']
export_wc = export_counts['whisker_only'] + export_counts['both']
export_corr = export_ec.corr(export_wc) if export_ec.std() > 0 and export_wc.std() > 0 else np.nan
summary = {
    'movie_path': str(output_path), 'frames': export_frame_count, 'fps': FPS,
    'duration_seconds': export_frame_count / FPS, 'bytes': output_path.stat().st_size,
    'eagle_spike_county_frames': export_e, 'whisker_spike_county_frames': export_w,
    'both_county_frames': export_b,
    'jaccard_agreement': export_b / max(export_e + export_w - export_b, 1),
    'timeseries_corr_eagle_vs_whisker_spike_counts': float(export_corr) if np.isfinite(export_corr) else None,
    'event_reference_start_UTC': str(EVENT_START), 'event_reference_end_UTC': str(EVENT_END),
    'movie_first_frame_UTC': str(frame_times[0]),
    'movie_last_frame_UTC': str(frame_times[export_frame_count - 1]),
    'spike_quantile': SPIKE_Q, 'eagle_min_delta_customers': EAGLE_MIN_DELTA,
    'whisker_rolling_window': WHISKER_WINDOW, 'frame_cadence': SPIKE_FREQ,
    'eagle_input_mode': EAGLE_INPUT_MODE, 'eagle_source': str(eagle_source),
    'whisker_files': [str(WHISKER_DIR / name) for name in EVENT_FILES],
    'county_source': str(COUNTY_SHP), 'ffmpeg': FFMPEG,
    'Whisker_unmapped_events': whisker_unmapped,
    'scope': 'mapped CONUS counties; exploratory event-referenced flags',
    'filling_policy': 'original hourly Eagle-I forward fill then zero fill; Whisker empty slots zero',
}
export_counts.to_csv(counts_path, index_label='time_UTC')
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
display(FileLink(str(output_path)))
if DISPLAY_VIDEO:
    display(Video(filename=str(output_path), embed=True, width=960))

In [ ]:
# Optional figure configuration
FIGURE_TIME_TEXT = "2024-05-11 02:00:00+00:00"
FIGURE_OUT = SPIKE_OUT.with_name("may2024_eagle_whisker_absolute_outages.png")
SAVE_FIGURE = False

from matplotlib.lines import Line2D

# Normalize the selected timestamp to UTC and the notebook's 15-minute cadence.
figure_time = pd.Timestamp(FIGURE_TIME_TEXT)
figure_time = (
    figure_time.tz_localize("UTC")
    if figure_time.tzinfo is None
    else figure_time.tz_convert("UTC")
)
figure_slot = figure_time.floor(SPIKE_FREQ)

# Assign the five source files to the three displayed Whisker families.
family_colors = {
    "Frequency": "#e31a1c",
    "Sags": "#1f78b4",
    "Swells": "#33a02c",
}
whisker_plot = whisker_events_df.copy()
whisker_plot["family"] = np.select(
    [
        whisker_plot["event_family"].str.startswith("Frequency"),
        whisker_plot["event_family"].str.startswith("Sags"),
        whisker_plot["event_family"].str.startswith("Swells"),
    ],
    ["Frequency", "Sags", "Swells"],
    default="Other",
)
whisker_plot["slot"] = whisker_plot["start_utc"].dt.floor(SPIKE_FREQ)

# Full-window time series: Eagle-I customers out and Whisker sensor-event sums.
eagle_total = eagle_wide.reindex(full_times).sum(axis=1)

whisker_series = (
    whisker_plot.loc[whisker_plot["family"].isin(family_colors)]
    .groupby(["slot", "family"])["sensors"]
    .sum()
    .unstack("family")
    .reindex(index=full_times, columns=family_colors.keys())
    .fillna(0.0)
)

# County-level absolute Eagle-I outage values for the selected snapshot.
if figure_slot not in eagle_wide.index:
    raise ValueError(f"No Eagle-I data grid row for {figure_slot}.")

county_values = eagle_wide.loc[figure_slot].reindex(conus["county_fips"]).fillna(0.0)
map_data = conus.copy()
map_data["customers_out"] = county_values.to_numpy(dtype=float)

# Whisker event dots for this snapshot; marker area scales with Sensors.
dots = whisker_plot.loc[
    (whisker_plot["slot"] == figure_slot)
    & whisker_plot["family"].isin(family_colors)
].copy()

max_sensors = max(float(dots["sensors"].max()), 1.0) if not dots.empty else 1.0
dots["marker_area"] = 12.0 + 90.0 * np.sqrt(dots["sensors"].clip(lower=0) / max_sensors)

fig, (ax_map, ax_ts) = plt.subplots(
    2, 1, figsize=(15, 11), gridspec_kw={"height_ratios": [3, 1.2]}
)

vmax = max(float(map_data["customers_out"].max()), 1.0)
map_data.plot(
    ax=ax_map,
    column="customers_out",
    cmap="magma",
    norm=mcolors.PowerNorm(gamma=0.35, vmin=0, vmax=vmax),
    linewidth=0.1,
    edgecolor="#bdbdbd",
    legend=True,
    legend_kwds={"label": "Eagle-I customers out", "shrink": 0.7},
)

for family, color in family_colors.items():
    subset = dots.loc[dots["family"] == family]
    if not subset.empty:
        ax_map.scatter(
            subset["lon"],
            subset["lat"],
            s=subset["marker_area"],
            c=color,
            edgecolors="white",
            linewidths=0.35,
            alpha=0.85,
            zorder=5,
        )

ax_map.set_xlim(-125, -66)
ax_map.set_ylim(24, 50)
ax_map.set_axis_off()
ax_map.set_title(
    f"Eagle-I customers out by county with Whisker events\\n"
    f"{figure_slot:%Y-%m-%d %H:%M} UTC"
)

family_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor=color, markeredgecolor="white",
        markersize=8, label=family,
    )
    for family, color in family_colors.items()
]
family_legend = ax_map.legend(
    handles=family_handles, title="Whisker event family", loc="lower left"
)
ax_map.add_artist(family_legend)

size_values = sorted(set([1.0, max_sensors]))
size_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="#777777", markeredgecolor="white",
        markersize=np.sqrt(12.0 + 90.0 * np.sqrt(value / max_sensors)),
        label=f"{value:g} sensors",
    )
    for value in size_values
]
ax_map.legend(handles=size_handles, title="Dot size", loc="lower right")

ax_ts.plot(
    eagle_total.index, eagle_total.values,
    color="black", linewidth=1.8, label="Eagle-I customers out",
)
ax_whisker = ax_ts.twinx()
for family, color in family_colors.items():
    ax_whisker.plot(
        whisker_series.index,
        whisker_series[family],
        color=color,
        linewidth=1.4,
        label=f"Whisker {family}",
    )

ax_ts.axvline(figure_slot, color="#555555", linestyle="--", linewidth=1)
ax_ts.set_ylabel("Eagle-I customers out")
ax_whisker.set_ylabel("Whisker sensor-event sum")
ax_ts.set_xlabel("Time (UTC)")
ax_ts.grid(alpha=0.2)
ax_ts.xaxis.set_major_formatter(
    mdates.DateFormatter("%m-%d\n%H:%M", tz=timezone.utc)
)

handles_left, labels_left = ax_ts.get_legend_handles_labels()
handles_right, labels_right = ax_whisker.get_legend_handles_labels()
ax_ts.legend(
    handles_left + handles_right,
    labels_left + labels_right,
    loc="upper left",
    ncol=2,
    fontsize=8,
)
fig.tight_layout()

if SAVE_FIGURE:
    FIGURE_OUT.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_OUT, dpi=180, bbox_inches="tight")
    print("Saved figure:", FIGURE_OUT)

plt.show()

In [ ]:
# Generate MP4: absolute Eagle-I county outages with Whisker event overlays

from matplotlib.lines import Line2D

OUTPUT_DIR = Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/output")
MOVIE_OUT = OUTPUT_DIR / "may2024_eagle_whisker_absolute_outages_may10_12.mp4"
MOVIE_OVERWRITE = False
MOVIE_DPI = DPI

if MOVIE_OUT.exists() and not MOVIE_OVERWRITE:
    raise FileExistsError(f"Output already exists: {MOVIE_OUT}")

movie_times = pd.date_range(SPIKE_START, SPIKE_END, freq=SPIKE_FREQ)

movie_families = {
    "Frequency": "#e31a1c",
    "Sags": "#1f78b4",
    "Swells": "#33a02c",
}
movie_events = whisker_events_df.copy()
movie_events["family"] = np.select(
    [
        movie_events["event_family"].str.startswith("Frequency"),
        movie_events["event_family"].str.startswith("Sags"),
        movie_events["event_family"].str.startswith("Swells"),
    ],
    ["Frequency", "Sags", "Swells"],
    default="Other",
)
movie_events["slot"] = movie_events["start_utc"].dt.floor(SPIKE_FREQ)
movie_events = movie_events.loc[movie_events["family"].isin(movie_families)].copy()

# Full reference-window series, displayed with the x-axis focused on the movie dates.
movie_eagle_series = eagle_wide.reindex(full_times).sum(axis=1)
movie_whisker_series = (
    movie_events.groupby(["slot", "family"])["sensors"]
    .sum()
    .unstack("family")
    .reindex(index=full_times, columns=movie_families.keys())
    .fillna(0.0)
)

# Explode multipart counties so each rendered polygon receives its county's value.
movie_polygons = (
    conus[["county_fips", "geometry"]]
    .explode(index_parts=False)
    .reset_index(drop=True)
)
movie_values_by_polygon = movie_polygons["county_fips"]

max_customers_out = max(float(eagle_wide.to_numpy().max()), 1.0)
outage_norm = mcolors.PowerNorm(
    gamma=0.35, vmin=0, vmax=max_customers_out
)

max_sensors = max(float(movie_events["sensors"].max()), 1.0)
movie_events["marker_area"] = (
    12.0
    + 90.0 * np.sqrt(movie_events["sensors"].clip(lower=0) / max_sensors)
)

fig = plt.figure(figsize=(15, 11))
ax_map = fig.add_axes([0.04, 0.34, 0.88, 0.61])
ax_ts = fig.add_axes([0.08, 0.07, 0.84, 0.19])

# Draw county geometry once; update its colors for each frame.
movie_polygons.plot(
    ax=ax_map, color="#f7f7f7", linewidth=0.1, edgecolor="#bdbdbd"
)
county_collection = ax_map.collections[-1]
if len(county_collection.get_paths()) != len(movie_polygons):
    plt.close(fig)
    raise ValueError("Rendered polygon order/count does not match county geometries.")

county_collection.set_cmap("magma")
county_collection.set_norm(outage_norm)
initial_values = (
    eagle_wide.loc[movie_times[0]]
    .reindex(movie_values_by_polygon)
    .fillna(0.0)
    .to_numpy(dtype=float)
)
county_collection.set_array(initial_values)

colorbar = fig.colorbar(
    county_collection, ax=ax_map, fraction=0.025, pad=0.015, shrink=0.75
)
colorbar.set_label("Eagle-I customers out")

ax_map.set_xlim(-125, -66)
ax_map.set_ylim(24, 50)
ax_map.set_axis_off()

# One scatter artist per event family, updated each frame.
event_scatters = {}
for family, color in movie_families.items():
    event_scatters[family] = ax_map.scatter(
        [], [], s=[], c=color, edgecolors="white",
        linewidths=0.35, alpha=0.85, zorder=5,
    )

family_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor=color, markeredgecolor="white",
        markersize=8, label=family,
    )
    for family, color in movie_families.items()
]
family_legend = ax_map.legend(
    handles=family_handles, title="Whisker event family", loc="lower left"
)
ax_map.add_artist(family_legend)

size_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="#777777", markeredgecolor="white",
        markersize=np.sqrt(12.0 + 90.0 * np.sqrt(value / max_sensors)),
        label=f"{value:g} sensors",
    )
    for value in sorted(set([1.0, max_sensors]))
]
ax_map.legend(handles=size_handles, title="Dot size", loc="lower right")

# Eagle-I uses the left axis; Whisker sensor-event sums use the right axis.
ax_ts.plot(
    movie_eagle_series.index,
    movie_eagle_series.values,
    color="black",
    linewidth=1.8,
    label="Eagle-I customers out",
)
ax_whisker = ax_ts.twinx()
for family, color in movie_families.items():
    ax_whisker.plot(
        movie_whisker_series.index,
        movie_whisker_series[family],
        color=color,
        linewidth=1.4,
        label=f"Whisker {family}",
    )

cursor = ax_ts.axvline(movie_times[0], color="#555555", linestyle="--", linewidth=1)
ax_ts.set_xlim(movie_times[0], movie_times[-1])
ax_ts.set_ylabel("Eagle-I customers out")
ax_whisker.set_ylabel("Whisker sensor-event sum")
ax_ts.set_xlabel("Time (UTC)")
ax_ts.grid(alpha=0.2)
ax_ts.xaxis.set_major_formatter(
    mdates.DateFormatter("%m-%d\n%H:%M", tz=timezone.utc)
)

left_handles, left_labels = ax_ts.get_legend_handles_labels()
right_handles, right_labels = ax_whisker.get_legend_handles_labels()
ax_ts.legend(
    left_handles + right_handles,
    left_labels + right_labels,
    loc="upper left",
    ncol=2,
    fontsize=8,
)

def draw_absolute_frame(i):
    timestamp = movie_times[i]

    values = (
        eagle_wide.loc[timestamp]
        .reindex(movie_values_by_polygon)
        .fillna(0.0)
        .to_numpy(dtype=float)
    )
    county_collection.set_array(values)

    frame_events = movie_events.loc[movie_events["slot"] == timestamp]
    for family, scatter in event_scatters.items():
        family_events = frame_events.loc[frame_events["family"] == family]
        if family_events.empty:
            scatter.set_offsets(np.empty((0, 2)))
            scatter.set_sizes([])
        else:
            scatter.set_offsets(family_events[["lon", "lat"]].to_numpy())
            scatter.set_sizes(family_events["marker_area"].to_numpy())

    ax_map.set_title(
        "Eagle-I customers out by county with Whisker events\n"
        f"{timestamp:%Y-%m-%d %H:%M} UTC"
    )
    cursor.set_xdata([timestamp, timestamp])
    return [county_collection, cursor, *event_scatters.values()]

animation = FuncAnimation(
    fig,
    draw_absolute_frame,
    frames=len(movie_times),
    interval=1000 / FPS,
    blit=False,
    repeat=False,
    cache_frame_data=False,
)

MOVIE_OUT.parent.mkdir(parents=True, exist_ok=True)
partial_movie = MOVIE_OUT.with_name(MOVIE_OUT.stem + ".partial.mp4")
writer = FFMpegWriter(
    fps=FPS,
    codec="libx264",
    bitrate=2400,
    metadata={"title": "Eagle-I absolute outages and Whisker events, May 10–12 2024"},
    extra_args=[
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
    ],
)

def movie_progress(current, total):
    if current % 48 == 0 or current + 1 == total:
        print(f"Encoding frame {current + 1}/{total}", flush=True)

try:
    animation.save(
        str(partial_movie),
        writer=writer,
        dpi=MOVIE_DPI,
        progress_callback=movie_progress,
    )
    partial_movie.replace(MOVIE_OUT)
finally:
    plt.close(fig)
    if partial_movie.exists():
        partial_movie.unlink()

print(f"Saved movie: {MOVIE_OUT}")
display(FileLink(str(MOVIE_OUT)))

In [ ]:
# Animate NOAA GeoE at 15-minute steps over May 10–12, using nearby native observations.
import xarray as xr
from matplotlib.animation import FuncAnimation, FFMpegWriter

GEOE_DIR = Path(
    "/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/"
    "NASA ROSES Disasters 2025-2027/data/space_weather/NOAA_geoE"
)
GEOE_MOVIE_OUT = OUTPUT_DIR / "noaa_geoe_may10_12_2024_15min.mp4"
GEOE_MOVIE_OVERWRITE = False
GEOE_VARIABLE = None  # Set to the GeoE data-variable name if discovery is ambiguous.
GEOE_TIME_TOLERANCE = pd.Timedelta("7min30s")

GEOE_START = pd.Timestamp("2024-05-10 00:00:00", tz="UTC")
GEOE_END = pd.Timestamp("2024-05-12 23:45:00", tz="UTC")
GEOE_FRAME_TIMES = pd.date_range(GEOE_START, GEOE_END, freq="15min")

if GEOE_MOVIE_OUT.exists() and not GEOE_MOVIE_OVERWRITE:
    raise FileExistsError(f"Output already exists: {GEOE_MOVIE_OUT}")

geoe_files = sorted(
    path for pattern in ("*.nc", "*.nc4", "*.cdf")
    for path in GEOE_DIR.rglob(pattern)
)
if not geoe_files:
    raise FileNotFoundError(f"No NetCDF/NetCDF4/CSV-format GeoE files found under {GEOE_DIR}")

# Open files individually, then combine their native time records.
geoe_parts = [xr.open_dataset(path) for path in geoe_files]
try:
    ds_geoe = xr.combine_by_coords(geoe_parts, combine_attrs="override")
except (ValueError, TypeError):
    ds_geoe = xr.concat(geoe_parts, dim="time", combine_attrs="override")

def _geoe_find_name(names, aliases):
    lookup = {str(name).lower(): name for name in names}
    return next((lookup[name] for name in aliases if name in lookup), None)

time_name = _geoe_find_name(
    list(ds_geoe.coords) + list(ds_geoe.dims),
    ["time", "datetime", "timestamp"],
)
lat_name = _geoe_find_name(
    list(ds_geoe.coords) + list(ds_geoe.data_vars),
    ["latitude", "lat", "glat", "mlat"],
)
lon_name = _geoe_find_name(
    list(ds_geoe.coords) + list(ds_geoe.data_vars),
    ["longitude", "lon", "glon", "mlon", "long"],
)

if time_name is None or lat_name is None or lon_name is None:
    raise ValueError(
        "Could not identify GeoE time/latitude/longitude fields. "
        f"Coordinates={list(ds_geoe.coords)}, variables={list(ds_geoe.data_vars)}"
    )

if GEOE_VARIABLE is None:
    possible = [
        name for name, da in ds_geoe.data_vars.items()
        if time_name in da.dims and name not in {lat_name, lon_name}
    ]
    if len(possible) != 1:
        raise ValueError(
            "Set GEOE_VARIABLE to the intended time-varying GeoE field. "
            f"Candidates={possible}; all variables={list(ds_geoe.data_vars)}"
        )
    GEOE_VARIABLE = possible[0]

if GEOE_VARIABLE not in ds_geoe.data_vars:
    raise KeyError(f"{GEOE_VARIABLE!r} is not a GeoE data variable.")

# Convert times to UTC, sort observations, and restrict the source data to the storm window.
native_times = pd.DatetimeIndex(
    pd.to_datetime(ds_geoe[time_name].values, errors="coerce", utc=True)
)
valid_time = ~native_times.isna()
native_times = native_times[valid_time]
order = np.argsort(native_times.asi8)
native_times = native_times[order]

field = ds_geoe[GEOE_VARIABLE]
if time_name not in field.dims:
    raise ValueError(f"{GEOE_VARIABLE!r} has no {time_name!r} dimension.")

field = field.isel({time_name: np.flatnonzero(valid_time)[order]})
in_window = (native_times >= GEOE_START - GEOE_TIME_TOLERANCE) & (
    native_times <= GEOE_END + GEOE_TIME_TOLERANCE
)
native_times = native_times[in_window]
field = field.isel({time_name: np.flatnonzero(in_window)})

if len(native_times) == 0:
    raise ValueError("No native GeoE observations overlap the requested movie period.")

# Use static coordinates for a geographic point/grid plot.
lat_da, lon_da = ds_geoe[lat_name], ds_geoe[lon_name]
if time_name in lat_da.dims:
    lat_da = lat_da.isel({time_name: 0})
if time_name in lon_da.dims:
    lon_da = lon_da.isel({time_name: 0})

lat_b, lon_b = xr.broadcast(lat_da, lon_da)
spatial_dims = list(lat_b.dims)

extra_dims = [dim for dim in field.dims if dim not in {time_name, *spatial_dims}]
if extra_dims:
    raise ValueError(
        f"{GEOE_VARIABLE!r} has additional dimensions {extra_dims}; "
        "select the appropriate component/level before animating."
    )

field = field.transpose(time_name, *spatial_dims)
lat_values = np.asarray(lat_b.transpose(*spatial_dims).values, dtype=float).ravel()
lon_values = np.asarray(lon_b.transpose(*spatial_dims).values, dtype=float).ravel()
valid_locations = np.isfinite(lat_values) & np.isfinite(lon_values)
lat_values, lon_values = lat_values[valid_locations], lon_values[valid_locations]

geo_values = np.asarray(field.values, dtype=float).reshape(len(native_times), -1)
geo_values = geo_values[:, valid_locations]
geo_values[~np.isfinite(geo_values)] = np.nan

# Match each 15-minute frame to its closest native observation; do not interpolate.
frame_indices = []
matched_times = []
for frame_time in GEOE_FRAME_TIMES:
    nearest = int(np.argmin(np.abs(native_times.asi8 - frame_time.value)))
    if abs(native_times[nearest] - frame_time) <= GEOE_TIME_TOLERANCE:
        frame_indices.append(nearest)
        matched_times.append(native_times[nearest])
    else:
        frame_indices.append(None)
        matched_times.append(None)

if not any(index is not None for index in frame_indices):
    raise ValueError(
        "No native GeoE observations fall within the 15-minute frame tolerance. "
        "Check the file cadence or adjust GEOE_TIME_TOLERANCE."
    )

finite_values = geo_values[np.isfinite(geo_values)]
if finite_values.size == 0:
    raise ValueError(f"No finite values found in {GEOE_VARIABLE!r}.")

# Symmetric limits suit signed electric-field values; percentiles reduce outlier domination.
color_limit = float(np.nanpercentile(np.abs(finite_values), 98))
if not np.isfinite(color_limit) or color_limit == 0:
    color_limit = float(np.nanmax(np.abs(finite_values))) or 1.0

units = field.attrs.get("units", "units not specified")
fig, ax = plt.subplots(figsize=(12, 7))
initial_index = next(index for index in frame_indices if index is not None)
initial_values = geo_values[initial_index]

scatter = ax.scatter(
    lon_values,
    lat_values,
    c=initial_values,
    s=12,
    cmap="RdBu_r",
    vmin=-color_limit,
    vmax=color_limit,
    linewidths=0,
)
colorbar = fig.colorbar(scatter, ax=ax, shrink=0.82)
colorbar.set_label(f"{GEOE_VARIABLE} ({units})")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2)
ax.set_title("NOAA GeoE | May 10–12, 2024")

def draw_geoe_frame(frame_number):
    requested_time = GEOE_FRAME_TIMES[frame_number]
    source_index = frame_indices[frame_number]

    if source_index is None:
        scatter.set_array(np.full(len(lon_values), np.nan))
        source_label = "no native observation within tolerance"
    else:
        scatter.set_array(geo_values[source_index])
        source_label = f"native observation: {matched_times[frame_number]:%Y-%m-%d %H:%M:%S UTC}"

    ax.set_title(
        f"NOAA GeoE | requested: {requested_time:%Y-%m-%d %H:%M UTC}\n"
        f"{source_label}"
    )
    return (scatter,)

animation_geoe = FuncAnimation(
    fig,
    draw_geoe_frame,
    frames=len(GEOE_FRAME_TIMES),
    interval=1000 / FPS,
    blit=False,
    repeat=False,
    cache_frame_data=False,
)

GEOE_MOVIE_OUT.parent.mkdir(parents=True, exist_ok=True)
geoe_partial = GEOE_MOVIE_OUT.with_name(GEOE_MOVIE_OUT.stem + ".partial.mp4")
geoe_writer = FFMpegWriter(
    fps=FPS,
    codec="libx264",
    bitrate=2400,
    metadata={"title": "NOAA GeoE, May 10–12 2024"},
    extra_args=[
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
    ],
)

try:
    animation_geoe.save(
        str(geoe_partial),
        writer=geoe_writer,
        dpi=DPI,
        progress_callback=movie_progress,
    )
    geoe_partial.replace(GEOE_MOVIE_OUT)
finally:
    plt.close(fig)
    if geoe_partial.exists():
        geoe_partial.unlink()
    for dataset in geoe_parts:
        dataset.close()

print(
    {
        "movie": str(GEOE_MOVIE_OUT),
        "requested_15min_frames": len(GEOE_FRAME_TIMES),
        "frames_with_native_data": sum(index is not None for index in frame_indices),
        "variable": GEOE_VARIABLE,
        "units": units,
        "native_observations_in_window": len(native_times),
    }
)
display(FileLink(str(GEOE_MOVIE_OUT)))